In [ ]:
from decouple import config


class PostgresSettings:
    """Lazy registry for Postgres / pgvector connection settings."""

    _user: str | None = None
    _password: str | None = None
    _host: str | None = None
    _port: str | None = None
    _db: str | None = None
    _schema_name: str | None = None
    _table_name: str | None = None
    _vector_size: int | None = None
    _pool_size: int | None = None
    _max_overflow: int | None = None
    _pool_timeout: int | None = None
    _hnsw_index_name: str | None = None
    _connection_string: str | None = None

    @classmethod
    def connection_string(cls, driver: str = "asyncpg") -> str:
        """Return shared connection string (built once, then reused)."""
        if cls._connection_string is None:
            cls._connection_string = (
                f"postgresql+{driver}://{cls.user()}:{cls.password()}"
                f"@{cls.host()}:{cls.port()}/{cls.db()}"
            )
        return cls._connection_string

    @classmethod
    def user(cls) -> str:
        if cls._user is None:
            cls._user = config("POSTGRES_USER")
        return cls._user

    @classmethod
    def password(cls) -> str:
        if cls._password is None:
            cls._password = config("POSTGRES_PASSWORD")
        return cls._password

    @classmethod
    def host(cls) -> str:
        if cls._host is None:
            cls._host = config("POSTGRES_HOST", default="localhost")
        return cls._host

    @classmethod
    def port(cls) -> str:
        if cls._port is None:
            cls._port = config("POSTGRES_PORT", default="6024")
        return cls._port

    @classmethod
    def db(cls) -> str:
        if cls._db is None:
            cls._db = config("POSTGRES_DB")
        return cls._db

    @classmethod
    def schema_name(cls) -> str:
        if cls._schema_name is None:
            cls._schema_name = config("SCHEMA_NAME", default="public")
        return cls._schema_name

    @classmethod
    def table_name(cls) -> str:
        if cls._table_name is None:
            cls._table_name = config("TABLE_NAME", default="vectorstore")
        return cls._table_name

    @classmethod
    def vector_size(cls) -> int:
        if cls._vector_size is None:
            cls._vector_size = config("VECTOR_SIZE", default=1024, cast=int)
        return cls._vector_size

    @classmethod
    def pool_size(cls) -> int:
        if cls._pool_size is None:
            cls._pool_size = config("POSTGRES_POOL_SIZE", default=5, cast=int)
        return cls._pool_size

    @classmethod
    def max_overflow(cls) -> int:
        if cls._max_overflow is None:
            cls._max_overflow = config("POSTGRES_MAX_OVERFLOW", default=10, cast=int)
        return cls._max_overflow

    @classmethod
    def pool_timeout(cls) -> int:
        if cls._pool_timeout is None:
            cls._pool_timeout = config("POSTGRES_POOL_TIMEOUT", default=30, cast=int)
        return cls._pool_timeout

    @classmethod
    def hnsw_index_name(cls) -> str:
        if cls._hnsw_index_name is None:
            cls._hnsw_index_name = config("HNSW_INDEX_NAME", default="my-hnsw-index")
        return cls._hnsw_index_name

    @classmethod
    def reset(cls) -> None:
        """Clear cache so the next accessors reload from env."""
        cls._user = None
        cls._password = None
        cls._host = None
        cls._port = None
        cls._db = None
        cls._schema_name = None
        cls._table_name = None
        cls._vector_size = None
        cls._pool_size = None
        cls._max_overflow = None
        cls._pool_timeout = None
        cls._hnsw_index_name = None
        cls._connection_string = None

In [ ]:
from langchain_core.embeddings import DeterministicFakeEmbedding

from common_module.vector_mill import PostgresSettings, VectorStorePipeline

print(PostgresSettings.connection_string())
print(
    PostgresSettings.schema_name(),
    PostgresSettings.table_name(),
    PostgresSettings.vector_size(),
)

In [ ]:
embedding = DeterministicFakeEmbedding(size=PostgresSettings.vector_size())

# First run: create_schema=True, create_table=True
await VectorStorePipeline.setup(
    embedding,
    create_schema=False,
    create_table=False,
)

In [ ]:
import uuid

from langchain_core.documents import Document

docs = [
    Document(
        id=str(uuid.uuid4()),
        page_content="Red Apple",
        metadata={"description": "red", "content": "1", "category": "fruit"},
    ),
    Document(
        id=str(uuid.uuid4()),
        page_content="Banana Cavendish",
        metadata={"description": "yellow", "content": "2", "category": "fruit"},
    ),
]

await VectorStorePipeline.add_documents(docs)

In [ ]:
await VectorStorePipeline.similarity_search("I'd like a fruit.")

In [ ]:
await VectorStorePipeline.similarity_search_with_score("red apple", k=2)

In [ ]:
# Upsert = add again with the same Document.id
# await VectorStorePipeline.get_by_ids([docs[0].id])
# await VectorStorePipeline.delete([docs[0].id])
# await VectorStorePipeline.apply_hnsw()   # after bulk load
# await VectorStorePipeline.reindex()      # optional later
# await VectorStorePipeline.close()        # notebook / app shutdown

In [1]:
from common_module.vector_mill.vector_pipeline import VectorStorePipeline

In [6]:
from langchain_core.embeddings import DeterministicFakeEmbedding
embedding = DeterministicFakeEmbedding(size=1024)
await VectorStorePipeline.setup(embedding=embedding)

In [8]:
import uuid
from langchain_core.documents import Document
import random
await VectorStorePipeline.setup(embedding=embedding)
await VectorStorePipeline.similarity_search_with_score("Hello, world!")


[(Document(id='821ed0fb-2573-4c46-a345-ff2f6c2d6da6', metadata={'llm': 'gpt-4o'}, page_content='Hello, world!'),
  0.0),
 (Document(id='76dd1a06-28c5-4307-a6c0-d3abfe2e98e6', metadata={'llm': 'gpt-4o'}, page_content='Hello, world!'),
  0.0),
 (Document(id='b802a1ef-24f1-4944-a9f8-52b489fdba36', metadata={'llm': 'gpt-4o-91'}, page_content='Hello, world! 7005'),
  0.8793094098996819),
 (Document(id='659d3556-464c-4635-96f0-81034d519ef7', metadata={'llm': 'gpt-4o-69'}, page_content='Hello, world! 1644'),
  0.8893498237947577)]

In [38]:
%time
await VectorStorePipeline.similarity_search_with_score("Hello, world!")

CPU times: user 3 μs, sys: 1e+03 ns, total: 4 μs
Wall time: 8.11 μs


[(Document(id='821ed0fb-2573-4c46-a345-ff2f6c2d6da6', metadata={'llm': 'gpt-4o'}, page_content='Hello, world!'),
  0.0),
 (Document(id='76dd1a06-28c5-4307-a6c0-d3abfe2e98e6', metadata={'llm': 'gpt-4o'}, page_content='Hello, world!'),
  0.0),
 (Document(id='b802a1ef-24f1-4944-a9f8-52b489fdba36', metadata={'llm': 'gpt-4o-91'}, page_content='Hello, world! 7005'),
  0.8793094098996819),
 (Document(id='659d3556-464c-4635-96f0-81034d519ef7', metadata={'llm': 'gpt-4o-69'}, page_content='Hello, world! 1644'),
  0.8893498237947577)]